SpaCy - sentence splitting

In [39]:
import spacy
import pandas as pd

reviews = pd.read_csv("training_set/Reviews.csv", encoding="cp1252")

nlp = spacy.load("en_core_web_sm")

def split_into_sentences(df, text_col="ReviewContent"):

    sentence_rows = []

    for review_id, text in df[text_col].dropna().items():

        doc = nlp(text)

        for sentence_id, sent in enumerate(doc.sents):

            sentence_text = sent.text.strip()

            if sentence_text:

                sentence_rows.append({

                    "review_id": review_id,

                    "sentence_id": sentence_id,

                    "sentence": sentence_text

                })

    return pd.DataFrame(sentence_rows)


sentences_df = split_into_sentences(reviews, text_col="ReviewContent")
for _, row in sentences_df.head(20).iterrows():
    print(f"Review {row['review_id']} | Sentence {row['sentence_id']}: {row['sentence']}")


Review 0 | Sentence 0: Good.
Review 0 | Sentence 1: It IS a page turner.
Review 0 | Sentence 2: You can read this book in one day, two at the most, and the plot drives the whole book.
Review 0 | Sentence 3: The unreliable narrators (there are two besides the main character) are as unlikable as they are unreliable, and there isn't a nice male in the book.
Review 0 | Sentence 4: Entirely plot driven; the characters are paper thin.
Review 0 | Sentence 5: You can figure out who-dunnit by the middle of the book.
Review 0 | Sentence 6: The ending is weak.
Review 0 | Sentence 7: I can't imagine what all the fuss is about, except that it is quick and there are lots of twists and turns, and you can't trust anyone to tell the truth.
Review 1 | Sentence 0: There are no words for how much I loathed this book.
Review 1 | Sentence 1: This was the first audiobook I ever listened to so not sure if something was lost in translation here, if I just disliked the narrators, or if it is truly the book I ha

1st system - VADER

In [51]:
import nltk
from nltk.sentiment import vader
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader_model = SentimentIntensityAnalyzer()
sentiment_scores = []

def vader_sentiment(sentence):
    score = vader_model.polarity_scores(sentence)["compound"]

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

for r_id, s_id, sent in sentences_df[["review_id", "sentence_id", "sentence"]].itertuples(index=False):
    sentiment_scores.append({

                    "review_id": r_id,

                    "sentence_id": s_id,

                    "sentence": sent,

                    "score": vader_model.polarity_scores(sent),

                    "label": vader_sentiment(sent)
                })
    
vader_df = pd.DataFrame(sentiment_scores)

for _, row in vader_df.head(20).iterrows():

    print(f"Review {row['review_id']} | Sentence {row['sentence_id']}: {row['sentence']}")

    print(f"Score: {row['score']}")

    print(f"Label: {row['label']}\n")


Review 0 | Sentence 0: Good.
Score: {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4404}
Label: positive

Review 0 | Sentence 1: It IS a page turner.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 2: You can read this book in one day, two at the most, and the plot drives the whole book.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 3: The unreliable narrators (there are two besides the main character) are as unlikable as they are unreliable, and there isn't a nice male in the book.
Score: {'neg': 0.089, 'neu': 0.911, 'pos': 0.0, 'compound': -0.3252}
Label: negative

Review 0 | Sentence 4: Entirely plot driven; the characters are paper thin.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | Sentence 5: You can figure out who-dunnit by the middle of the book.
Score: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Label: neutral

Review 0 | S